# Sprint 5

## Install PySpark

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [2]:
import os
import platform

if platform.system() == "Windows":
    # Point to Java 17 explicitly — required for PySpark on Windows
    os.environ["JAVA_HOME"] = r"C:\Users\bhoom\AppData\Local\Programs\Microsoft\jdk-17.0.18.8-hotspot"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
    print("Windows: Java 17 path set to", os.environ["JAVA_HOME"])
else:
    print("Non-Windows: no fix needed")

Non-Windows: no fix needed


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 15:35:25 WARN Utils: Your hostname, Aradhanas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.5.138 instead (on interface en0)
26/04/18 15:35:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 15:35:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


26/04/18 15:35:29 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:582)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:809)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:725)
	at java.base/java.util.ServiceLoader$3.next(ServiceLoader.java:1397)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

Shuffle partitions: 8


## Import the funtions and create data path

In [4]:
from pathlib import Path
from urllib.request import urlretrieve # to download data if not already present

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

# Path for MIMIC-IV data
DATA_DIR = Path("data/MIMIC-IV/hosp")


## Dataframes

In [5]:
# -------------------------------------------------------
# Visits dataframe (Ara) 
# -------------------------------------------------------

# reading the source tables
admissions = spark.read.csv(str(DATA_DIR / "admissions.csv.gz"), header=True, inferSchema=True)
patients = spark.read.csv(str(DATA_DIR / "patients.csv.gz"), header=True, inferSchema=True)

# reading sprint3 output timeline
pre_bc_system_timeline = spark.read.csv(
    "out/evidence/pre_bc_symptom_timeline.csv",
    header=True,
    inferSchema=True
)

# keping visit-level rows of interest as specified on trello card 
visits_of_interest = (
    pre_bc_system_timeline
    .select("subject_id", "hadm_id", "row_type")
    .dropna(subset=["subject_id", "hadm_id", "row_type"])
    .dropDuplicates(["subject_id", "hadm_id", "row_type"])
)

# mapping sprint3 labels to the required visit labels
visits_labeled = (
    visits_of_interest
    .withColumn(
        "visit_type",
        F.when(
            F.col("row_type") == "BC_FIRST_DX",
            F.lit("BC_FIRST_DIAGNOSIS")
        ).otherwise(F.lit("SYMPTOM"))
    )
)

# here lies below the visits dataframe 
visits = (
    visits_labeled
    .join(
        admissions.select("subject_id", "hadm_id", "admittime", "race"),
        on=["subject_id", "hadm_id"],
        how="inner"
    )
    .join(
        patients.select("subject_id", "gender", "anchor_age", "anchor_year"),
        on="subject_id",
        how="inner"
    )
    .withColumn("admit_day", F.to_date("admittime"))
    .withColumn(
        "age",
        (
            F.col("anchor_age") + (F.year("admit_day") - F.col("anchor_year"))
        ).cast("int")
    )
    .select(
        F.col("subject_id").cast("int").alias("subject_id"),
        F.col("hadm_id").cast("int").alias("hadm_id"),
        F.col("race").cast("string").alias("race"),
        F.col("gender").cast("string").alias("gender"),
        F.col("visit_type").cast("string").alias("visit_type"),
        F.col("age").cast("int").alias("age"),
        F.col("admit_day")
    )
    .dropDuplicates(["subject_id", "hadm_id", "visit_type"])
)

visits.printSchema()
visits.show(20, truncate=False)


26/04/18 15:35:29 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/MIMIC-IV/hosp/admissions.csv.gz.
java.lang.UnsupportedOperationException: getSubject is supported only if a security manager is allowed
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:347)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3888)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3878)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3666)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:289)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:541)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	

UnsupportedOperationException: getSubject is supported only if a security manager is allowed

In [ ]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

EVIDENCE_DIR = Path("out/evidence")
# DATA_DIR is already defined above as Path("data/MIMIC-IV/hosp")

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)


In [ ]:
# -------------------------------------------------------
# AGE and DATE Transformations (Ara)
# -------------------------------------------------------

# subtask 1: Frequency of Age at Diagnosis


print("Subtask 1: Frequency of Age at Diagnosis")

bc_visits = visits.filter(F.col("visit_type") == "BC_FIRST_DIAGNOSIS")

bc_buckets = bc_visits.withColumn(
    "de_obfs_age_bucket",
    F.when(F.col("age") < 30, "<30")
     .when((F.col("age") >= 30) & (F.col("age") <= 40), "30-40")
     .when((F.col("age") >= 41) & (F.col("age") <= 55), "41-55")
     .when((F.col("age") >= 56) & (F.col("age") <= 70), "56-70")
     .when((F.col("age") >= 71) & (F.col("age") <= 85), "71-85")
     .otherwise("85+")
)

age_frequency = (
    bc_buckets
    .groupBy("de_obfs_age_bucket")
    .count()
    .orderBy("de_obfs_age_bucket")
)

age_frequency.show(truncate=False)


# subtask 2: Days Prior to Diagnosis


# Use the timeline here because it preserves symptom/diagnosis event order
symptoms = pre_bc_system_timeline.filter(F.col("row_type") == "SYMPTOM")
bc = pre_bc_system_timeline.filter(F.col("row_type") == "BC_FIRST_DX")

symptom_counts = (
    symptoms
    .groupBy("subject_id")
    .count()
    .withColumnRenamed("count", "num_of_symptoms")
)
# the window...
from pyspark.sql.window import Window
the_window = Window.partitionBy("subject_id").orderBy(F.col("admittime").desc())

last_symptoms = (
    symptoms
    .withColumn("rank", F.row_number().over(the_window))
    .filter(F.col("rank") == 1)
    .select("subject_id", F.col("admittime").alias("last_symptom_time"))
)

first_bc = (
    bc
    .select("subject_id", F.col("admittime").alias("bc_time"))
)

time_difference = (
    last_symptoms
    .join(first_bc, on="subject_id", how="inner")
    .join(symptom_counts, on="subject_id", how="inner")
    .withColumn(
        "days_before_dx",
        F.datediff("bc_time", "last_symptom_time")
    )
)

time_difference = time_difference.withColumn(
    "bucket",
    F.when(F.col("num_of_symptoms") == 1, "1")
     .when(F.col("num_of_symptoms") == 2, "2")
     .otherwise("3+")
)

stats_bucket = (
    time_difference
    .groupBy("bucket")
    .agg(
        F.mean("days_before_dx").alias("mean"),
        F.stddev("days_before_dx").alias("stddev"),
        F.max("days_before_dx").alias("max"),
        F.min("days_before_dx").alias("min"),
        F.expr("percentile_approx(days_before_dx, 0.5)").alias("median")
    )
    .orderBy("bucket")
)

stats_bucket.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------


## Clean up
Stop spark session when done

In [ ]:
# Uncomment when you are completely done:

# spark.stop()

26/04/18 15:35:41 ERROR Utils: Process ArraySeq(getconf, PAGESIZE) exited with code 143: 
26/04/18 15:35:41 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/fx/k1wgdlt121731v5ss1w3dpv40000gn/T/blockmgr-47768052-2512-48ca-824d-8690567c943e. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/fx/k1wgdlt121731v5ss1w3dpv40000gn/T/blockmgr-47768052-2512-48ca-824d-8690567c943e
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.storage.DiskBlo